# Download Day Ahead Price and Load Forecast Data

In this notebook the day ahead price and load forecast is downloaded from ENTSO-E Transparency Platform using API. Also, the data is processed at a uniform hourly granularity and data quality checks are made (missing values).


In [1]:
from entsoe import EntsoePandasClient
import pandas as pd

ENTSO-E Codes for API:
* https://www.entsoe.eu/Documents/EDI/Library/Core/entso-e-code-list-v36r0.pdf
* https://pypi.org/project/entsoe-py/0.3.3/?utm_source=chatgpt.com

In [2]:
# Load API key
def read_cdsapi_key(path="../api_keys.txt"):
    with open(path) as f:
        for line in f:
            if line.startswith("ENTSOE_API_KEY="):
                return line.strip().split("=", 1)[1]
    raise KeyError("ENTSOE_API_KEY not found")

ENTSOE_API_KEY = read_cdsapi_key()

client = EntsoePandasClient(api_key=ENTSOE_API_KEY)

# set bidding zone
zone = "DE_LU"

# Load Day Ahead Price and Load Forecast

In [3]:
# set time range
start = pd.Timestamp("2022-07-01 00:00", tz="Europe/Berlin")
end   = pd.Timestamp("2026-01-01 00:00", tz="Europe/Berlin")

# half yearly intervals, else the API fails
dates = pd.date_range(start, end, freq="6MS", tz="Europe/Berlin")


price_parts = []
load_parts = []

for s, e in zip(dates[:-1], dates[1:]):
    price_parts.append(client.query_day_ahead_prices(zone, start=s, end=e))
    load_parts.append(client.query_load_forecast(zone, start=s, end=e))
    print(f"{s.date()} → {e.date()} geladen")

da_price = pd.concat(price_parts).sort_index()
load_fc  = pd.concat(load_parts).sort_index()

2022-07-01 → 2023-01-01 geladen
2023-01-01 → 2023-07-01 geladen
2023-07-01 → 2024-01-01 geladen
2024-01-01 → 2024-07-01 geladen
2024-07-01 → 2025-01-01 geladen
2025-01-01 → 2025-07-01 geladen
2025-07-01 → 2026-01-01 geladen


### Check for missing data

In [4]:
# check for missing timestamps in load data
expected_index_load = pd.date_range(start=pd.Timestamp("2022-07-01 00:00", tz="Europe/Berlin"), end=pd.Timestamp("2026-01-01 00:00", tz="Europe/Berlin"), freq="15min")
missing_timestamps_load = expected_index_load.difference(load_fc.index)
missing_timestamps_load

DatetimeIndex(['2023-10-29 00:00:00+02:00', '2023-10-29 00:15:00+02:00',
               '2023-10-29 00:30:00+02:00', '2023-10-29 00:45:00+02:00',
               '2024-10-27 00:00:00+02:00', '2024-10-27 00:15:00+02:00',
               '2024-10-27 00:30:00+02:00', '2024-10-27 00:45:00+02:00',
               '2026-01-01 00:00:00+01:00'],
              dtype='datetime64[ns, Europe/Berlin]', freq=None)

In [5]:
# some missing values -> linear imputation
load_fc = load_fc.sort_index().reindex(expected_index_load)
load_fc_imputed = load_fc.interpolate(method="linear")

In [6]:
# check for missing timestamps in price data
# differentiate between hourly and 15min data, as the granularity changes after 2025-10-01
expected_index_price = pd.date_range(start=pd.Timestamp("2022-07-01 00:00", tz="Europe/Berlin"), end=pd.Timestamp("2025-09-30 23:00", tz="Europe/Berlin"), freq="h")
expected_index_price2 = pd.date_range(start=pd.Timestamp("2025-10-01 00:00", tz="Europe/Berlin"), end=pd.Timestamp("2026-01-01 00:00", tz="Europe/Berlin"), freq="15min")
missing_timestamps_price = expected_index_price.append(expected_index_price2).difference(da_price.index)
missing_timestamps_price

DatetimeIndex([], dtype='datetime64[ns, Europe/Berlin]', freq=None)

### Aggregate to hourly granularity

* From 2025-10-01 the prices are in quarter hourely granularity -> build mean price per hour to get consistent granularity
* Load forecast is all over given in quarter hourly granularity -> build mean load forecast per hour to get consistent granularity


In [7]:
# resample to hourly data
da_price_hrly = da_price.resample("1h").mean()
load_fc_hrly = load_fc_imputed.resample("1h").mean()

# merge to dataframe
price_df_hrly = da_price_hrly.squeeze().to_frame("da_price")
load_df_hrly  = load_fc_hrly.squeeze().to_frame("load_fc")
df = price_df_hrly.join(load_df_hrly, how="outer")

### Save data

In [8]:
df.to_parquet("entsoe_price_and_loadfc.parquet", compression="zstd")